1. Load data

In [15]:
import json
import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [16]:
def load_corpus(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_claims(path):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    return [json.loads(l) for l in content.splitlines() if l.strip()]

In [17]:
corpus = load_corpus("data/data/corpus.jsonl")
claims_train = load_claims("data/data/claims_train.jsonl")
claims_dev   = load_claims("data/data/claims_dev.jsonl")
claims_test  = load_claims("data/data/claims_test.jsonl")
corpus_by_id = {p["doc_id"]: p for p in corpus}

print(f"Corpus papers: {len(corpus)}")
print(f"Train claims: {len(claims_train)} | Dev claims: {len(claims_dev)} | Test claims: {len(claims_test)}")

Corpus papers: 5183
Train claims: 809 | Dev claims: 300 | Test claims: 300


 2. Build (claim, sentence, label) training pairs from gold evidence annotations

In [18]:
def build_pairs(claims, corpus_by_id):
    rows = []
    for claim in claims:
        evidence = claim.get("evidence", {})
        if not evidence:
            continue
        for doc_id_str, ev_list in evidence.items():
            doc_id = int(doc_id_str)
            paper = corpus_by_id.get(doc_id)
            if paper is None:
                continue
            gold = set()
            for ev in ev_list:
                gold.update(ev["sentences"])
            for idx, sentence in enumerate(paper["abstract"]):
                rows.append({"claim": claim["claim"], "sentence": sentence,
                             "doc_id": doc_id, "sentence_idx": idx,
                             "label": 1 if idx in gold else 0})
    return rows


In [19]:
train_rows = build_pairs(claims_train, corpus_by_id)
dev_rows   = build_pairs(claims_dev, corpus_by_id)
print(f"Train pairs: {len(train_rows)} (positive: {sum(r['label'] for r in train_rows)})")
print(f"Dev pairs:   {len(dev_rows)} (positive: {sum(r['label'] for r in dev_rows)})")

Train pairs: 5494 (positive: 1025)
Dev pairs:   2031 (positive: 366)


3. TF-IDF baseline features

In [8]:
vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))
vectorizer.fit([r["claim"] for r in train_rows] + [r["sentence"] for r in train_rows])

def word_overlap(a, b):
    wa, wb = set(a.lower().split()), set(b.lower().split())
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

In [9]:
def featurize(rows):
    claim_vecs = vectorizer.transform([r["claim"] for r in rows])
    sent_vecs  = vectorizer.transform([r["sentence"] for r in rows])
    sims = np.array([cosine_similarity(claim_vecs[i], sent_vecs[i])[0][0] for i in range(len(rows))])
    overlaps = np.array([word_overlap(r["claim"], r["sentence"]) for r in rows])
    positions = np.array([r["sentence_idx"] for r in rows])
    lengths = np.array([len(r["sentence"].split()) for r in rows])
    X = np.column_stack([sims, overlaps, positions, lengths])
    y = np.array([r["label"] for r in rows])
    return X, y

X_train, y_train = featurize(train_rows)
X_dev, y_dev = featurize(dev_rows)

4. Train the classifier

In [10]:
clf = LogisticRegression(class_weight="balanced", max_iter=1000)
clf.fit(X_train, y_train)

print("=== Dev set report (per-sentence) ===")
print(classification_report(y_dev, clf.predict(X_dev), target_names=["Not evidence", "Evidence"]))

=== Dev set report (per-sentence) ===
              precision    recall  f1-score   support

Not evidence       0.91      0.66      0.77      1665
    Evidence       0.31      0.70      0.43       366

    accuracy                           0.67      2031
   macro avg       0.61      0.68      0.60      2031
weighted avg       0.80      0.67      0.71      2031



5. Save the trained model 

In [11]:
joblib.dump({"model": clf, "vectorizer": vectorizer}, "rationale_selector.joblib")
print("Saved -> rationale_selector.joblib")

Saved -> rationale_selector.joblib


6. Inference function

In [12]:
def select_evidence(claim_text, retrieved_paper, clf=clf, vectorizer=vectorizer, threshold=0.5):
    abstract = retrieved_paper["abstract"]
    rows = [{"claim": claim_text, "sentence": s, "sentence_idx": i} for i, s in enumerate(abstract)]
    X, _ = featurize([{**r, "label": 0} for r in rows])
    probs = clf.predict_proba(X)[:, 1]

    selected = [
        {"sentence_idx": i, "sentence": abstract[i], "probability": float(probs[i])}
        for i in range(len(abstract)) if probs[i] >= threshold
    ]
    selected.sort(key=lambda x: x["probability"], reverse=True)
    return selected

In [13]:
example_claim = next(c for c in claims_dev if c.get("evidence"))
example_doc_id = int(list(example_claim["evidence"].keys())[0])
example_paper = corpus_by_id[example_doc_id]

result = select_evidence(example_claim["claim"], example_paper)
print("Claim:", example_claim["claim"])
for r in result:
    print(f"  [{r['sentence_idx']}] (p={r['probability']:.2f}) {r['sentence']}")

Claim: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
  [7] (p=0.88) In conclusion, uncommon or rare genetic variants can easily create synthetic associations that are credited to common variants, and this possibility requires careful consideration in the interpretation and follow up of GWAS signals.
  [6] (p=0.80) We also illustrate the behavior of synthetic associations in real datasets by showing that rare causal mutations responsible for both hearing loss and sickle cell anemia create genome-wide significant synthetic associations, in the latter case extending over a 2.5-Mb interval encompassing scores of "blocks" of associated variants.
  [2] (p=0.74) We propose as an alternative explanation that variants much less common than the associated one may create "synthetic associations" by occurring, stochastically, more often in association with one of the alleles at the common site ve

In [ ]:
with open("retrieved_docs.json") as f:
    retrieved = json.load(f)
claim_id = "5"
doc_ids_for_claim = retrieved["dev"][claim_id]
print(doc_ids_for_claim)

claim_text = next(c["claim"] for c in claims_dev if str(c["id"]) == claim_id)
paper = corpus_by_id[doc_ids_for_claim[0]]

selected_evidence = select_evidence(claim_text, paper)
selected_evidence

[13734012, 18617259, 17333231, 42240424, 695938]


[{'sentence_idx': 7,
  'sentence': 'CONCLUSIONS This study corroborates previous studies and suggests a high prevalence of infection with abnormal PrP, indicating vCJD carrier status in the population compared with the 177 vCJD cases to date.',
  'probability': 0.6831416191315844},
 {'sentence_idx': 0,
  'sentence': 'OBJECTIVES To carry out a further survey of archived appendix samples to understand better the differences between existing estimates of the prevalence of subclinical infection with prions after the bovine spongiform encephalopathy epizootic and to see whether a broader birth cohort was affected, and to understand better the implications for the management of blood and blood products and for the handling of surgical instruments.   \n',
  'probability': 0.6659877314703065},
 {'sentence_idx': 6,
  'sentence': 'Genetic testing of the positive specimens for the genotype at PRNP codon 129 revealed a high proportion that were valine homozygous compared with the frequency in the 